In [1]:
%pip install lightgbm xgboost catboost


  Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl (3.6 MB)
  Using cached xgboost-3.0.2-py3-none-manylinux_2_28_x86_64.whl (253.9 MB)
  Using cached catboost-1.2.8-cp310-cp310-manylinux2014_x86_64.whl (99.2 MB)
  Using cached nvidia_nccl_cu12-2.26.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (318.1 MB)
  Using cached plotly-6.1.2-py3-none-any.whl (16.3 MB)
  Using cached graphviz-0.20.3-py3-none-any.whl (47 kB)
  Using cached narwhals-1.41.0-py3-none-any.whl (357 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import pandas as pd
import joblib
from datetime import datetime
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from src.config import *
from src.utils import get_latest_file
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# Charger les données
dataset_path = get_latest_file(DATA_FINAL_CLEANED_DATASET_DIR)
df = pd.read_csv(dataset_path)

# Préparation des données
target = 'IS_WIN'
drop_cols = ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID', 'SEASON', 'GAME_DATE']
features = [col for col in df.columns if col not in drop_cols + [target]]
df = df.dropna(subset=features)

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Définition des modèles
estimators = [
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('lgbm', LGBMClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                          subsample=0.7, colsample_bytree=0.7,
                          random_state=42, eval_metric='logloss',
                          n_jobs=-1, use_label_encoder=False)),
    ('cat', CatBoostClassifier(n_estimators=200, learning_rate=0.05,
                               depth=6, verbose=0, random_state=42)),
    ('et', ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('hgb', HistGradientBoostingClassifier(max_iter=200, random_state=42)),
    ('mlp', MLPClassifier(hidden_layer_sizes=(50, ), max_iter=300, random_state=42)),
    # ('knn', KNeighborsClassifier(n_neighbors=5))
]

final_estimator = LogisticRegression(solver='lbfgs', max_iter=5000)


# print("ROC AUC avec cross-validation (5 folds) :")
# for name, model in estimators:
#     scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
#     print(f"{name} mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Pipeline avec scaler + stacking
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator,
    passthrough=True,
    cv=5,
    n_jobs=-1
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', stack)
])

# Entraînement
pipeline.fit(X_train, y_train)

# Évaluation
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))

# Sauvegarde
today = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

os.makedirs(DATA_MODELS_DIR, exist_ok=True)

stacking_model_path = os.path.join(DATA_MODELS_DIR, f"stacking_model_{today}.joblib")
joblib.dump(pipeline,stacking_model_path)
print(f"Model saved in {stacking_model_path}")


/opt/conda/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [18:40:33] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[18:45:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[18:45:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[18:45:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[18:45:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[18:45:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.



[LightGBM] [Info] Number of positive: 20547, number of negative: 20490[LightGBM] [Info] Number of positive: 20547, number of negative: 20491

[LightGBM] [Info] Number of positive: 20547, number of negative: 20490
[LightGBM] [Info] Number of positive: 20548, number of negative: 20490
[LightGBM] [Info] Number of positive: 20547, number of negative: 20491
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.398511 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 28574
[LightGBM] [Info] Number of data points in the train set: 41037, number of used features: 148
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500694 -> initscore=0.002778
[LightGBM] [Info] Start training from score 0.002778
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.467156 seconds.
You can set `force_row_wise=true` to remove the overhe

/opt/conda/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:702: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


ROC AUC: 0.7190297856817146
Model saved in data/models/stacking_model_2025-05-28_18-52-31.joblib
